In [2]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName("DeltaLakeColab")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

**Initial Load**

In [4]:
passengers_day1 = [
(101,"Rahul Sharma","Hyderabad","Economy","India"),
(102,"Priya Reddy","Bangalore","Business","India"),
(103,"Amit Kumar","Mumbai","Economy","India"),
(104,"Sneha Patel","Delhi","Premium Economy","India"),
(105,"Farhan Ali","Chennai","Economy","India")
]
columns = [
"passenger_id",
"passenger_name",
"city",
"travel_class",
"country"
]
df_day1 = spark.createDataFrame(
    passengers_day1,
    columns
)

**Incremental Load**

In [5]:
passengers_day2 = [
(102,"Priya Reddy","Bangalore","First Class","India"),
(104,"Sneha Patel","Hyderabad","Premium Economy","India"),
(106,"Neha Singh","Pune","Economy","India"),
(107,"Arjun Verma","Kochi","Business","India")
]
df_day2 = spark.createDataFrame(
    passengers_day2,
    columns
)

**Create Delta**

In [6]:
# 1
delta_path = "/content/passengers_delta"

df_day1.write.format("delta").mode("overwrite").save(delta_path)

In [7]:
# 2
df_delta = spark.read.format("delta").load(delta_path)

df_delta.show()

+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         103|    Amit Kumar|   Mumbai|        Economy|  India|
|         104|   Sneha Patel|    Delhi|Premium Economy|  India|
|         105|    Farhan Ali|  Chennai|        Economy|  India|
|         101|  Rahul Sharma|Hyderabad|        Economy|  India|
|         102|   Priya Reddy|Bangalore|       Business|  India|
+------------+--------------+---------+---------------+-------+



In [8]:
# 3
df_delta.count()

5

In [9]:
# 4
df_delta.show()

+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         103|    Amit Kumar|   Mumbai|        Economy|  India|
|         104|   Sneha Patel|    Delhi|Premium Economy|  India|
|         105|    Farhan Ali|  Chennai|        Economy|  India|
|         101|  Rahul Sharma|Hyderabad|        Economy|  India|
|         102|   Priya Reddy|Bangalore|       Business|  India|
+------------+--------------+---------+---------------+-------+



In [10]:
# 5
from delta.tables import DeltaTable

deltaTable = DeltaTable.forPath(
    spark,
    delta_path
)

deltaTable.history().show(truncate=False)

+-------+-----------------------+------+--------+---------+--------------------------------------+----+--------+---------+-----------+--------------+-------------+-----------------------------------------------------------+------------+-----------------------------------+
|version|timestamp              |userId|userName|operation|operationParameters                   |job |notebook|clusterId|readVersion|isolationLevel|isBlindAppend|operationMetrics                                           |userMetadata|engineInfo                         |
+-------+-----------------------+------+--------+---------+--------------------------------------+----+--------+---------+-----------+--------------+-------------+-----------------------------------------------------------+------------+-----------------------------------+
|0      |2026-06-18 04:14:32.604|NULL  |NULL    |WRITE    |{mode -> Overwrite, partitionBy -> []}|NULL|NULL    |NULL     |NULL       |Serializable  |false        |{numFiles -> 2, nu

**Merge / Upsert**

In [12]:
# 6,7,8
from delta.tables import DeltaTable

deltaTable = DeltaTable.forPath(
    spark,
    delta_path
)

deltaTable.alias("target") \
.merge(
    df_day2.alias("source"),
    "target.passenger_id = source.passenger_id"
) \
.whenMatchedUpdateAll() \
.whenNotMatchedInsertAll() \
.execute()

In [16]:
# 7 verify
spark.read.format("delta") \
.load(delta_path) \
.filter("passenger_id IN (102,104)") \
.show(truncate=False)

+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|city     |travel_class   |country|
+------------+--------------+---------+---------------+-------+
|102         |Priya Reddy   |Bangalore|First Class    |India  |
|104         |Sneha Patel   |Hyderabad|Premium Economy|India  |
+------------+--------------+---------+---------------+-------+



In [15]:
# 8 verify
spark.read.format("delta") \
.load(delta_path) \
.filter("passenger_id IN (106,107)") \
.show(truncate=False)

+------------+--------------+-----+------------+-------+
|passenger_id|passenger_name|city |travel_class|country|
+------------+--------------+-----+------------+-------+
|106         |Neha Singh    |Pune |Economy     |India  |
|107         |Arjun Verma   |Kochi|Business    |India  |
+------------+--------------+-----+------------+-------+



In [17]:
# 9
df = spark.read.format("delta").load(delta_path)
df.filter("passenger_id=102").show()

+------------+--------------+---------+------------+-------+
|passenger_id|passenger_name|     city|travel_class|country|
+------------+--------------+---------+------------+-------+
|         102|   Priya Reddy|Bangalore| First Class|  India|
+------------+--------------+---------+------------+-------+



In [18]:
#10
df.filter("passenger_id=106").show()

+------------+--------------+----+------------+-------+
|passenger_id|passenger_name|city|travel_class|country|
+------------+--------------+----+------------+-------+
|         106|    Neha Singh|Pune|     Economy|  India|
+------------+--------------+----+------------+-------+



**Time Travel**

In [19]:
# 11
version0 = spark.read \
    .format("delta") \
    .option("versionAsOf",0) \
    .load(delta_path)

version0.show()

+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         103|    Amit Kumar|   Mumbai|        Economy|  India|
|         104|   Sneha Patel|    Delhi|Premium Economy|  India|
|         105|    Farhan Ali|  Chennai|        Economy|  India|
|         101|  Rahul Sharma|Hyderabad|        Economy|  India|
|         102|   Priya Reddy|Bangalore|       Business|  India|
+------------+--------------+---------+---------------+-------+



In [20]:
# 12
latest = spark.read \
    .format("delta") \
    .load(delta_path)

latest.show()

+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         101|  Rahul Sharma|Hyderabad|        Economy|  India|
|         102|   Priya Reddy|Bangalore|    First Class|  India|
|         103|    Amit Kumar|   Mumbai|        Economy|  India|
|         104|   Sneha Patel|Hyderabad|Premium Economy|  India|
|         105|    Farhan Ali|  Chennai|        Economy|  India|
|         106|    Neha Singh|     Pune|        Economy|  India|
|         107|   Arjun Verma|    Kochi|       Business|  India|
+------------+--------------+---------+---------------+-------+



In [21]:
# 13
latest.exceptAll(version0).show()

+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         104|   Sneha Patel|Hyderabad|Premium Economy|  India|
|         107|   Arjun Verma|    Kochi|       Business|  India|
|         106|    Neha Singh|     Pune|        Economy|  India|
|         102|   Priya Reddy|Bangalore|    First Class|  India|
+------------+--------------+---------+---------------+-------+



In [22]:
# 14
# Version 0
spark.read \
.format("delta") \
.option("versionAsOf",0) \
.load(delta_path) \
.filter("passenger_id=102") \
.show()
# Latest
spark.read \
.format("delta") \
.load(delta_path) \
.filter("passenger_id=102") \
.show()

+------------+--------------+---------+------------+-------+
|passenger_id|passenger_name|     city|travel_class|country|
+------------+--------------+---------+------------+-------+
|         102|   Priya Reddy|Bangalore|    Business|  India|
+------------+--------------+---------+------------+-------+

+------------+--------------+---------+------------+-------+
|passenger_id|passenger_name|     city|travel_class|country|
+------------+--------------+---------+------------+-------+
|         102|   Priya Reddy|Bangalore| First Class|  India|
+------------+--------------+---------+------------+-------+



In [23]:
# 15
# Version 0
spark.read \
.format("delta") \
.option("versionAsOf",0) \
.load(delta_path) \
.filter("passenger_id=104") \
.show()
# Latest
spark.read \
.format("delta") \
.load(delta_path) \
.filter("passenger_id=104") \
.show()

+------------+--------------+-----+---------------+-------+
|passenger_id|passenger_name| city|   travel_class|country|
+------------+--------------+-----+---------------+-------+
|         104|   Sneha Patel|Delhi|Premium Economy|  India|
+------------+--------------+-----+---------------+-------+

+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         104|   Sneha Patel|Hyderabad|Premium Economy|  India|
+------------+--------------+---------+---------------+-------+



**Maintenance**

In [25]:
# 16
deltaTable = DeltaTable.forPath(
    spark,
    delta_path
)

deltaTable.delete("passenger_id = 105")

spark.read.format("delta") \
.load(delta_path) \
.show()

+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         101|  Rahul Sharma|Hyderabad|        Economy|  India|
|         102|   Priya Reddy|Bangalore|    First Class|  India|
|         103|    Amit Kumar|   Mumbai|        Economy|  India|
|         104|   Sneha Patel|Hyderabad|Premium Economy|  India|
|         106|    Neha Singh|     Pune|        Economy|  India|
|         107|   Arjun Verma|    Kochi|       Business|  India|
+------------+--------------+---------+---------------+-------+



In [26]:
# 17
deltaTable.history().show(truncate=False)

+-------+-----------------------+------+--------+---------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+--------+---------+-----------+--------------+-------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+-----------------------------------+

In [28]:
# 18
spark.conf.set(
    "spark.databricks.delta.retentionDurationCheck.enabled",
    "false"
)

deltaTable.vacuum(0)

spark.read.format("delta") \
.load(delta_path) \
.orderBy("passenger_id") \
.show(truncate=False)

+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|city     |travel_class   |country|
+------------+--------------+---------+---------------+-------+
|101         |Rahul Sharma  |Hyderabad|Economy        |India  |
|102         |Priya Reddy   |Bangalore|First Class    |India  |
|103         |Amit Kumar    |Mumbai   |Economy        |India  |
|104         |Sneha Patel   |Hyderabad|Premium Economy|India  |
|106         |Neha Singh    |Pune     |Economy        |India  |
|107         |Arjun Verma   |Kochi    |Business       |India  |
+------------+--------------+---------+---------------+-------+

